# Φάση 3

## Imports

In [1]:
#! pip install elasticsearch

import os
import numpy as np
import pandas as pd

from elasticsearch import Elasticsearch
from sentence_transformers import SentenceTransformer
import faiss

## Ρυθμίσεις paths / ElasticSearch


In [2]:
QUERIES_CSV = "queries.csv"

INDEX_NAME = "ir2025_index"  # Δημιουργήθηκε στην Φάση 1
ES_HOST = "http://localhost:9200"

# Outputs
RUNS_DIR = "runs_phase3"
os.makedirs(RUNS_DIR, exist_ok=True)

## Φόρτωση queries (queries.csv)


In [3]:
queries_df = pd.read_csv(QUERIES_CSV)

def get_qid_text(row):
    return str(row["ID"]).strip(), str(row["Text"])

queries_df.head()

,ID,Text
0,Q01,EUTRAVEL Optimodal European Travel Ecosystem E...
1,Q02,Track And Know Big Data for Mobility Tracking ...
2,Q03,"SELIS, Towards a Shared European Logistics Int..."
3,Q04,TYPHON Polyglot and Hybrid Persistence Archite...
4,Q05,CHARIOT Cognitive Heterogeneous Architecture f...


## Φόρτωση ElasticSearch + Transformer model (all-MiniLM-L6-v2)


In [4]:
es = Elasticsearch(ES_HOST)

# ίδιο μοντέλο με Φάση 2
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")  # 384 dims

## Εκτέλεση ερωτήματος στο ευρετήριο της ElasticSearch, που δημιουργήθηκε στην Φάση 1, και επιστροφή των Ν πιο σχετικών κειμένων

In [5]:
def bm25_candidates(query_text: str, N: int = 200):
    '''
    Επιστρέφει λίστα από (doc_id, doc_text, bm25_score) για τα top-N του Elastic.
    πεδία: id, text (όπως στη Φάση 1).
    '''
    body = {
        "size": N,
        "_source": ["id", "text"],
        "query": {"match": {"text": query_text}}
    }
    resp = es.search(index=INDEX_NAME, body=body)
    hits = resp["hits"]["hits"]

    out = []
    for h in hits:
        src = h["_source"]
        doc_id = int(src["id"])
        doc_text = src.get("text", "")
        score = float(h["_score"])
        out.append((doc_id, doc_text, score))
    return out

## Δημιουργία ενθέσεων (embeddings) για τα Ν κείμενα και FAISS index πάνω στα Ν κείμενα (FAISS-only ranking)


In [6]:
def faiss_rerank(query_text: str, candidates, k: int = 20, batch_size: int = 64):
    '''
    candidates: λίστα (doc_id, doc_text, bm25_score)
    επιστρέφει λίστα top-k από dicts με {doc_id, faiss_score, bm25_score, rank}
    '''
    if len(candidates) == 0:
        return []

    doc_ids = [c[0] for c in candidates]
    doc_texts = [c[1] for c in candidates]
    bm25_scores = np.array([c[2] for c in candidates], dtype=np.float32)

    # embeddings (L2-normalized από το μοντέλο, όπως στη Φάση 2)
    doc_emb = model.encode(
        doc_texts,
        batch_size=batch_size,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    q_emb = model.encode(
        [query_text],
        batch_size=1,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    dim = doc_emb.shape[1]
    index = faiss.IndexFlatIP(dim)     # IP πάνω σε normalized vectors = cosine
    index.add(doc_emb)

    k_eff = min(k, len(doc_ids))
    scores, idxs = index.search(q_emb, k_eff)

    results = []
    for rank, (score, idx) in enumerate(zip(scores[0], idxs[0]), start=1):
        results.append({
            "doc_id": int(doc_ids[int(idx)]),
            "faiss_score": float(score),
            "bm25_score": float(bm25_scores[int(idx)]),
            "rank": int(rank)
        })
    return results

## Συνάρτηση που τρέχει όλα τα queries και παράγει run files για το trec_eval

In [7]:
def write_run_file(run_path: str, run_name: str, results_by_qid: dict):
    '''
    results_by_qid: {qid: [ {doc_id, score..., rank}, ... ] }
    Γράφει TREC format: qid Q0 docid rank score runname
    '''
    with open(run_path, "w", encoding="utf-8") as f:
        for qid, res_list in results_by_qid.items():
            for r in res_list:
                docid = r["doc_id"]
                rank = r["rank"]
                score = r.get("final_score", r.get("faiss_score", 0.0))
                f.write(f"{qid} Q0 {docid} {rank} {score:.6f} {run_name}\n")

## Εκτελούμε: για FAISS, για τα τρία k: 20, 30, 50

In [8]:
# FAISS-only runs + write runfiles
for k in (20, 30, 50):
    results_by_qid = {}

    for idx, row in queries_df.iterrows():
        qid, qtext = get_qid_text(row)

        # 1) BM25 candidates from Elastic
        cands = bm25_candidates(qtext, N=200)

        # 2) FAISS rerank
        res = faiss_rerank(qtext, cands, k=k, batch_size=64)
        results_by_qid[qid] = res

    out_path = os.path.join(RUNS_DIR, f"phase3_faiss_k{k}.txt")
    run_name = f"phase3_faiss_k{k}"
    write_run_file(out_path, run_name, results_by_qid)
    print("Έτοιμο:", out_path)

Έτοιμο: runs_phase3\phase3_faiss_k20.txt
Έτοιμο: runs_phase3\phase3_faiss_k30.txt
Έτοιμο: runs_phase3\phase3_faiss_k50.txt


# <center>Συνδυασμός βαθμολογιών (Fusion)

## Σταθμισμένος μέσος όρος BM25 score και cosine similarity (Fusion)

In [9]:
def fusion_rerank(query_text: str, candidates, k: int = 20, alpha: float = 0.7, batch_size: int = 64):
    '''
    Υπολογίζει cosine similarity με FAISS στα candidates και μετά υπολογίζει τον σταθμισμένο μέσο όρο (fusion) με BM25 (min-max normalized).
    Επιστρέφει top-k με βάση final_score.
    '''
    if len(candidates) == 0:
        return []

    doc_ids   = [c[0] for c in candidates]
    doc_texts = [c[1] for c in candidates]
    bm25      = np.array([c[2] for c in candidates], dtype=np.float32)

    # Embeddings (L2-normalized από SentenceTransformer, όπως Φάση 2)
    doc_emb = model.encode(
        doc_texts,
        batch_size=batch_size,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    q_emb = model.encode(
        [query_text],
        batch_size=1,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    dim = doc_emb.shape[1]
    index = faiss.IndexFlatIP(dim)  # IP πάνω σε normalized vectors = cosine
    index.add(doc_emb)

    # Θέλουμε cosine scores για ΟΛΑ τα σχετικά κείμενα ώστε να κάνουμε σωστό normalization
    scores, idxs = index.search(q_emb, len(doc_ids))  # (1, N)

    # scores/idxs είναι ήδη σε φθίνουσα σειρά score
    # Φτιάχνουμε vector cos όπου cos[i] είναι το cosine score του i-ου σχετικού κειμένου
    cos = np.zeros(len(doc_ids), dtype=np.float32)
    cos[idxs[0]] = scores[0].astype(np.float32)

    def minmax(x: np.ndarray):
        mn, mx = float(np.min(x)), float(np.max(x))
        return (x - mn) / (mx - mn + 1e-9)

    bm25_n = minmax(bm25)
    cos_n  = minmax(cos)

    final = alpha * bm25_n + (1.0 - alpha) * cos_n

    # Ταξινόμηση με βάση το final score
    order = np.argsort(-final)
    k_eff = min(k, len(doc_ids))

    results = []
    for rank, i in enumerate(order[:k_eff], start=1):
        results.append({
            "doc_id": int(doc_ids[i]),
            "final_score": float(final[i]),
            "faiss_score": float(cos[i]),
            "bm25_score": float(bm25[i]),
            "rank": int(rank)
        })

    return results

## Εκτελούμε: για Fusion (alpha=0.7), για τα τρία k: 20, 30, 50

In [10]:
# Fusion runs + write runfiles
alpha = 0.7

for k in (20, 30, 50):
    results_by_qid = {}

    for idx, row in queries_df.iterrows():
        qid, qtext = get_qid_text(row)

        # 1) BM25 candidates from Elastic
        cands = bm25_candidates(qtext, N=200)

        # 2) Fusion rerank
        res = fusion_rerank(qtext, cands, k=k, alpha=alpha, batch_size=64)
        results_by_qid[qid] = res

    out_path = os.path.join(RUNS_DIR, f"phase3_fusion_k{k}.txt")
    run_name = f"phase3_fusion_k{k}"
    write_run_file(out_path, run_name, results_by_qid)
    print("Έτοιμο:", out_path)

Έτοιμο: runs_phase3\phase3_fusion_k20.txt
Έτοιμο: runs_phase3\phase3_fusion_k30.txt
Έτοιμο: runs_phase3\phase3_fusion_k50.txt
